# Path A, P23 — design builder, L1 parameter recovery, target-code timing (Oct 29, 2026)

**Spec (as given; not changed after seeing results):**

TASK: (1) a design builder, (2) parameter recovery for L1, (3) target-code timing.
Do NOT compare simulated targets with human targets. Never load Ring/CT (never pass allow_test).

0. Decisions on the open questions
   - Keep hazard.py's old-CL feature definitions in the simulator: at_end = |k| >= 9; first move = first change of k; tau_r resets at a reversal's first k-step.
   - Keep CC features on the 0.033 s tick snapshot, as built.
   - Write both decisions in sim.py's docstring.
1. Design builder (src/slider/design.py)
   - design(run_set) -> one row per real round: group, dataset (EL/CL/CC), role, topology, round id; graph + seat->node map; interface (lab v09 / old CL / CC); step d; speed; dt = d / speed; time limit T; lab flag; whole_second flag.
   - Speed rule: lab rounds: that round's own speed from interface.py (modal pace x step), falling back to the file's modal speed; CC rounds: the file's cap (the file's modal speed); whole-second old-CL files: 0.2 units/s.
   - Run sets: "L3" = load_runs("outcome", roles={"train","val"}) (targets 1-7); "L5" = load_runs("behavior", roles={"train","val"}, keep_bot=False, keep_incomplete=False) (targets 8-10); "G6train" = load_runs("behavior") (the G-train rounds of the hazard table, bot and incomplete runs included; they are simulated all-human).
   - Checks: report rounds per (topology, dataset) cell for each set; L5 is a subset of L3, and L3 minus L5 = the whole-second rounds only.
   - simulate_design(design, preset, R, seed) -> (runs, table), using each round's own dataset, role, interface, speed and T.
2. Parameter recovery (L1)
   - Simulate L1 on the G6train design, R = 2, seed 573. Report the simulated bins and reversals next to G-train's (336,073 bins; 6,073 reversals).
   - Build a hazard table from the simulated runs with exactly the real pipeline: hazard.build_hazard_table, b = 0.25, m = 2, CC through the 0.1 s lab clock.
   - Refit M2' with fit.fit: R0 structure (shared weights + lab term), all cells. PASS: every weight within 2 model SEs of the value used in the simulation.
   - Re-estimate lam_stop, lam_resume, lam_start per interface on the simulated runs, with the P17 code (motion.py). PASS: each within its 95% CI in results/tables/p17_rates.csv.
   - If exactly one weight misses: rerun once with seed 574 and report both runs.
   - Never change the simulator or the pipeline to make recovery pass. Report any failure as it is.
   - Output: results/tables/p23_recovery.csv with columns true, recovered, SE, z, pass.
3. Target-code timing (planning only)
   - Simulate L1 on the L3 design, R = 1.
   - Time each of these: the simulation; targets.train_targets on the L3 runs; reversal_rate_table, agreement_at_reversal_table and crossing_time_table on the L5 runs.
   - Do NOT print, save or compare any target values.
4. Report: files and tests (the full suite passes); the recovery table and verdicts; the timings; any spec gaps and how they were filled. Commit.

## Choices the spec leaves open

Fixed in `design.py` and this notebook before any recovery number was computed.

- **File speeds** are the P7e values in `results/tables/interface_table.csv`. A lab round's own speed is `interface.step_and_pace(interface.moves(run))`, the same rule as `hazard.build_hazard_table` (its `round_speed`).
- **Seat -> node map**: the topology's graph in `models/apem.GRAPHS`, matched to the round's adjacency with networkx (`design.seat_map`). Any valid map gives the same simulated round.
- **Simulated groups**: copy r of log file g is group `sim<r>_<g>`, with the same rounds (waves) as g. The file name keeps g's version tag, so `interface.py` reads the interface back.
- **Seeds**: `SeedSequence(seed)` is split into one stream per design row, then into a simulation stream and a logger stream.
- **Recovery fit**: `fit.fit(ht, [A, abs_s, at_end, time_at_end_log1p, time_since_rev_log1p], lab_interact=False)` on every simulated bin. Every simulated round copies a G-train round, so no split filter is applied. The weights used in the simulation are `sim.coefs("M2'")`, the exact values in `p11_coefs.csv`. z = (recovered - true) / model SE; PASS if |z| <= 2.
- **Rates**: `motion.rates_table(runs)` as in P17 (1,000 resamples, seed 573), for lab_v09, old_cl and cc; the pooled "shared" row is not used, because the simulator uses per-interface rates. PASS if the estimate lies within the P17 95% CI. The SE and z columns for rates use the Poisson SE sqrt(events) / exposure; they are descriptive, and the verdict uses the CI.
- **Rerun rule**: if exactly one of the 7 weights misses, everything in part 2 is rerun once with seed 574, and both runs go into `p23_recovery.csv` (column `seed`). Otherwise no rerun.
- **Timing**: the L5 runs are the simulated L3 runs of the L5 rounds (L3 minus the whole-second rounds). `train_targets` and the three tables run with their default settings. Results are kept in variables only; nothing is displayed.

In [1]:
import time

import numpy as np
import pandas as pd

from slider import config, design, fit, hazard, motion, sim, targets

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 220)

## 1. Designs: rounds per (topology, dataset) cell

In [2]:
designs = {name: design.design(name) for name in design.SETS}       # about 15 s
cells = pd.concat({name: d.groupby(["topology", "dataset"]).size() for name, d in designs.items()}, axis=1)
cells = cells.fillna(0).astype(int)
cells.loc[("all", ""), :] = cells.sum()
cells

L3     L5  G6train
topology  dataset                       
Full      CC        36.0   36.0     48.0
          CL        63.0   54.0     33.0
Hierarchy CC        40.0   40.0     50.0
          CL        61.0   53.0     33.0
          EL        51.0   51.0     33.0
Star      CC        39.0   39.0     47.0
all                290.0  273.0    244.0

In [3]:
l3, l5 = designs["L3"], designs["L5"]
print("L5 is a subset of L3:", set(l5["round_id"]) <= set(l3["round_id"]))
print("L3 minus L5 = the whole-second rounds:",
      set(l3["round_id"]) - set(l5["round_id"]) == set(l3.loc[l3["whole_second"], "round_id"]),
      f"({int(l3['whole_second'].sum())} rounds)")
print("roles:", {name: sorted(d["role"].unique()) for name, d in designs.items()})

L5 is a subset of L3: True
L3 minus L5 = the whole-second rounds: True (17 rounds)
roles: {'L3': ['train', 'val'], 'L5': ['train', 'val'], 'G6train': ['train']}


Interface, speed (units/s) and dt (s) of the rounds in each set:

In [4]:
pd.concat({name: d.groupby(["interface", "whole_second", "speed", "dt"]).size() for name, d in designs.items()},
          axis=1).fillna(0).astype(int)

L3   L5  G6train
interface whole_second speed dt                         
cc        False        2.94  0.034014    9    9       17
                       3.03  0.033003   97   97      102
                       3.33  0.030030    9    9       17
lab_v09   False        0.50  0.200000    4    4        3
                       1.00  0.100000  102  102       68
                       2.00  0.050000   23   23       16
old_cl    False        0.20  0.100000   29   29       12
          True         0.20  0.100000   17    0        0
cc        False        3.23  0.030960    0    0        9

## 2. Parameter recovery: L1 on the G6train design, R = 2

In [5]:
FEATS = ["A", "abs_s", "at_end", "time_at_end_log1p", "time_since_rev_log1p"]    # M2'
TRUE = sim.coefs("M2'")                                   # the weights the simulation uses
P17 = pd.read_csv(config.P17_RATES).set_index(["interface", "rate"])
G_TRAIN = {"bins": 336073, "reversals": 6073}              # G-train hazard table, all cells (P8, P11)


def recover(seed, R=2):
    """Simulate L1 on G6train, rebuild the hazard table, refit M2', re-estimate the P17 rates.
    Returns (one row per parameter, counts)."""
    start = time.time()
    runs, table = design.simulate_design(designs["G6train"], "L1", R=R, seed=seed)
    ht, dropped = hazard.build_hazard_table(runs, b=0.25, m=2)     # CC on the 0.1 s lab clock
    model = fit.fit(ht, FEATS, False)                              # R0: shared weights + lab
    rates = motion.rates_table(runs).set_index(["interface", "rate"])
    rows = []
    for term, true in TRUE.items():
        b, se = model.params[term], model.bse[term]
        rows.append({"seed": seed, "part": "M2' weight", "parameter": term, "true": true, "recovered": b, "SE": se,
                     "z": (b - true) / se, "ci_low": np.nan, "ci_high": np.nan, "pass": abs(b - true) <= 2 * se})
    for (iface, rate), p in P17.loc[motion.INTERFACES].iterrows():
        r = rates.loc[(iface, rate)]
        se = np.sqrt(r["events"]) / r["exposure_s"]
        rows.append({"seed": seed, "part": "rate", "parameter": f"{iface} {rate}", "true": p["estimate"],
                     "recovered": r["estimate"], "SE": se, "z": (r["estimate"] - p["estimate"]) / se,
                     "ci_low": p["ci_low"], "ci_high": p["ci_high"],
                     "pass": p["ci_low"] <= r["estimate"] <= p["ci_high"]})
    counts = {"seed": seed, "rounds": len(runs), "bins": len(ht), "reversals": int(ht["y_rev"].sum()),
              "dropped mid-event bins": len(dropped), "converged": model.mle_retvals["converged"],
              "minutes": round((time.time() - start) / 60, 1)}
    return pd.DataFrame(rows), counts


rec, counts = recover(573)                                        # several minutes
runs_of = [counts]
misses = rec[(rec["part"] == "M2' weight") & ~rec["pass"]]
if len(misses) == 1:
    rec2, counts2 = recover(574)
    rec = pd.concat([rec, rec2], ignore_index=True)
    runs_of.append(counts2)
rec.to_csv(config.P23_RECOVERY, index=False)
pd.DataFrame(runs_of)

,seed,rounds,bins,reversals,dropped mid-event bins,converged,minutes
0,573,488,1413764,34610,17252,True,3.6


Simulated bins and reversals next to G-train's (per copy = total / R):

In [6]:
n = pd.DataFrame(runs_of).set_index("seed")[["bins", "reversals"]]
side = pd.DataFrame({"G-train (P8)": G_TRAIN})
for seed, row in n.iterrows():
    side[f"simulated, seed {seed}, R = 2"] = row
    side[f"per copy, seed {seed}"] = row / 2
side = side.T
side["reversals per 1,000 bins"] = 1000 * side["reversals"] / side["bins"]
side.round(1)

,bins,reversals,"reversals per 1,000 bins"
G-train (P8),336073.0,6073.0,18.1
"simulated, seed 573, R = 2",1413764.0,34610.0,24.5
"per copy, seed 573",706882.0,17305.0,24.5


Recovery table (`results/tables/p23_recovery.csv`):

In [7]:
with pd.option_context("display.float_format", "{:.4f}".format):
    display(rec)

,seed,part,parameter,true,recovered,SE,z,ci_low,ci_high,pass
0,573,M2' weight,const,-3.1941,-3.1957,0.0252,-0.0637,NaN,NaN,True
1,573,M2' weight,lab,0.0578,0.0179,0.0117,-3.4262,NaN,NaN,False
2,573,M2' weight,A,-2.1074,-2.1089,0.0225,-0.0672,NaN,NaN,True
3,573,M2' weight,abs_s,1.0476,1.0637,0.0382,0.4208,NaN,NaN,True
4,573,M2' weight,at_end,0.4446,0.4033,0.0248,-1.6611,NaN,NaN,True
5,573,M2' weight,time_at_end_log1p,-0.3774,-0.3988,0.0112,-1.9023,NaN,NaN,True
6,573,M2' weight,time_since_rev_log1p,-0.2093,-0.1697,0.0122,3.2399,NaN,NaN,False
7,573,rate,lab_v09 lam_stop,3.0393,2.4806,0.0121,-46.0077,2.4854,3.5327,False
8,573,rate,lab_v09 lam_resume,1.7681,1.6136,0.0079,-19.6151,1.4773,2.0651,True
9,573,rate,lab_v09 lam_start,0.3171,0.3059,0.0088,-1.2799,0.2675,0.3714,True


In [8]:
for seed, r in rec.groupby("seed"):
    w, q = r[r["part"] == "M2' weight"], r[r["part"] == "rate"]
    print(f"seed {seed}: M2' weights {int(w['pass'].sum())} of {len(w)} pass "
          f"({'PASS' if w['pass'].all() else 'FAIL'}); rates {int(q['pass'].sum())} of {len(q)} pass "
          f"({'PASS' if q['pass'].all() else 'FAIL'})")
    for _, x in r[~r["pass"]].iterrows():
        print(f"   miss: {x['parameter']}: true {x['true']:.4f}, recovered {x['recovered']:.4f}, z = {x['z']:.2f}")

seed 573: M2' weights 5 of 7 pass (FAIL); rates 6 of 9 pass (FAIL)
   miss: lab: true 0.0578, recovered 0.0179, z = -3.43
   miss: time_since_rev_log1p: true -0.2093, recovered -0.1697, z = 3.24
   miss: lab_v09 lam_stop: true 3.0393, recovered 2.4806, z = -46.01
   miss: old_cl lam_stop: true 1.7603, recovered 1.4919, z = -25.99
   miss: cc lam_stop: true 10.3500, recovered 8.2786, z = -61.02


**Rate conversion (descriptive; added after a smoke test on 3 rounds per interface showed lam_stop low in every interface; it changes no verdict).** The simulator turns each rate into a per-tick probability p = 1 - exp(-lam dt). The P17 estimator counts events per tick of exposure (stops / (steps x dt), resumes / halted ticks x dt), so it estimates p / dt. From this alone, recovered / true would be (1 - exp(-lam dt)) / (lam dt), at each interface's most common dt.

In [9]:
dts = designs["G6train"].groupby("interface")["dt"].agg(lambda x: x.mode()[0])
r = rec[(rec["part"] == "rate") & (rec["seed"] == 573)].copy()
r["interface"] = r["parameter"].str.split().str[0]
r["dt (modal)"] = r["interface"].map(dts)
x = r["true"] * r["dt (modal)"]
r["expected ratio"] = (1 - np.exp(-x)) / x
r["observed ratio"] = r["recovered"] / r["true"]
r[["parameter", "true", "recovered", "dt (modal)", "expected ratio", "observed ratio", "pass"]].round(3)

,parameter,true,recovered,dt (modal),expected ratio,observed ratio,pass
7,lab_v09 lam_stop,3.039,2.481,0.100,0.862,0.816,False
8,lab_v09 lam_resume,1.768,1.614,0.100,0.917,0.913,True
9,lab_v09 lam_start,0.317,0.306,0.100,0.984,0.965,True
10,old_cl lam_stop,1.760,1.492,0.100,0.917,0.848,False
11,old_cl lam_resume,1.736,1.704,0.100,0.918,0.982,True
12,old_cl lam_start,0.080,0.085,0.100,0.996,1.065,True
13,cc lam_stop,10.350,8.279,0.033,0.847,0.800,False
14,cc lam_resume,2.549,2.448,0.033,0.959,0.960,True
15,cc lam_start,0.204,0.203,0.033,0.997,0.996,True


## 3. Target-code timing (planning only)

No target value is printed, saved or compared: the results stay in variables.

In [10]:
timing = {}
start = time.perf_counter()
runs3, table3 = design.simulate_design(designs["L3"], "L1", R=1, seed=573)
timing["simulate L1 on L3, R = 1"] = time.perf_counter() - start

keep = ~designs["L3"]["whole_second"].to_numpy()                 # R = 1: runs3[i] is design row i
assert list(designs["L3"].loc[keep, "round_id"]) == list(designs["L5"]["round_id"])
runs5 = [run for run, k in zip(runs3, keep) if k]

start = time.perf_counter()
_tt = targets.train_targets(runs3, table3)
timing["train_targets (L3 runs)"] = time.perf_counter() - start
for name, f in [("reversal_rate_table", targets.reversal_rate_table),
                ("agreement_at_reversal_table", targets.agreement_at_reversal_table),
                ("crossing_time_table", targets.crossing_time_table)]:
    start = time.perf_counter()
    _out = f(runs5)
    timing[f"{name} (L5 runs)"] = time.perf_counter() - start
print(len(runs3), "L3 runs,", len(runs5), "L5 runs")
pd.Series(timing, name="seconds").round(1).to_frame()

290 L3 runs, 273 L5 runs


,seconds
"simulate L1 on L3, R = 1",139.2
train_targets (L3 runs),36.5
reversal_rate_table (L5 runs),4.4
agreement_at_reversal_table (L5 runs),221.7
crossing_time_table (L5 runs),29.0
